# Grounded Carbon-Market QA: train + evaluate (Kaggle)
Settings → Accelerator **GPU T4 x2**, Internet **ON**. Add-ons → Secrets: `GROQ_API_KEY`, `HF_TOKEN`. Attach the uploaded `carbon-grounded` dataset. Then **Run All**.

In [ ]:
import glob, os, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/config.py', recursive=True)[0])
shutil.copytree(src, '/kaggle/working/proj', dirs_exist_ok=True)
os.chdir('/kaggle/working/proj')
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ('GROQ_API_KEY', 'HF_TOKEN'):
    try: os.environ[k] = s.get_secret(k)
    except Exception as e: print('missing secret', k)
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
print(os.listdir('.'))

In [ ]:
!pip install -q aligntune unsloth rank_bm25 sentence-transformers litellm python-dotenv auditkit 2>&1 | tail -3
!python -c "import aligntune, unsloth, torch; print('ok', torch.cuda.get_device_name(0))"

In [ ]:
# 5-minute smoke test: proves AlignTune + Unsloth + data format work before the long run
!python scripts/05_train.py --max-samples 40 --epochs 1

In [ ]:
# Full training (~1 h on T4). If Unsloth fails above, add: --backend trl
!python scripts/05_train.py

In [ ]:
# 3 systems on the hand-checked test set
!python scripts/06_generate.py

In [ ]:
# Scoring (Groq judge + AuditKIT). If the daily Groq limit is hit, rerun this cell tomorrow: results are cached.
!python scripts/07_evaluate.py
from IPython.display import Image, Markdown, display
display(Markdown(open('outputs/results/report.md').read())); display(Image('outputs/results/chart.png'))

In [ ]:
# Save everything to the notebook output (download from the Output tab)
!cd outputs && zip -qr /kaggle/working/results.zip adapter preds results

In [ ]:
# Demo with a public link (stop the cell to end it)
!pip install -q gradio
!python app.py